In [1]:
import json
import pandas as pd
import folium
import geopandas as gpd
from folium.plugins import HeatMap

In [25]:
gdf = gpd.read_file("hangjeongdong_경기도.geojson")

gdf_gyeonggi = gdf[gdf["sggnm"].str.contains("양주시|의정부|포천|동두천")].copy()

dlvry_df = pd.read_csv("양주시전달지.csv")
# dlvry_df = pd.read_csv("경기도전달지.csv")


gdf_gyeonggi.head()

,OBJECTID,adm_nm,adm_cd,adm_cd2,sgg,sido,sidonm,sggnm,geometry
90,1246,경기도 의정부시 의정부1동,3103051,4115051000,41150,41,경기도,의정부시,"MULTIPOLYGON (((127.04608 37.75377, 127.04743 ..."
91,1247,경기도 의정부시 의정부2동,3103052,4115052000,41150,41,경기도,의정부시,"MULTIPOLYGON (((127.04675 37.73872, 127.04615 ..."
92,1248,경기도 의정부시 의정부3동,3103053,4115053000,41150,41,경기도,의정부시,"MULTIPOLYGON (((127.04675 37.73872, 127.05314 ..."
93,1249,경기도 의정부시 호원1동,3103055,4115054500,41150,41,경기도,의정부시,"MULTIPOLYGON (((127.05042 37.72367, 127.05018 ..."
94,1250,경기도 의정부시 장암동,3103056,4115056100,41150,41,경기도,의정부시,"MULTIPOLYGON (((127.07343 37.71992, 127.07379 ..."


In [26]:
heat_data = dlvry_df[['lat', 'lon', 'ord_cnt']].values.tolist()
heat_data_filtered = [
    [lat, lon, cnt]
    for lat, lon, cnt in heat_data
    if cnt > 3
]

gradient = {
    0.2: 'blue',
    0.4: 'cyan',
    0.6: 'lime',
    0.8: 'yellow',
    1.0: 'red'
}

In [ ]:
m = folium.Map(
    location=[37.82151762350629, 127.09344094168883],  # PPC 위치 
    zoom_start=12,
    tiles="cartodbpositron"   # OpenStreetMap  cartodbpositron
)

folium.GeoJson(
    gdf_gyeonggi,
    name="경기도 행정동",
    control=False,
    style_function=lambda x: {
        "fillColor": "lightgrey",
        "color": "black",
        "weight": 1.5,
        "fillOpacity": 0.6,
    },
    tooltip=folium.GeoJsonTooltip(
    fields=["adm_nm"],        # 행정동 이름 컬럼
    aliases=["행정동"],
    sticky=True               # 마우스 따라다니게
    )
).add_to(m)

# 새로운 PPC 위치 
folium.Marker(
    location=[37.82151762350629, 127.09344094168883]
    , icon=folium.Icon(icon='house', prefix='fa', color='lightred')
    ).add_to(m)

HeatMap(
    data=heat_data,    # 데이터 12월1일~21일, 주문수 3건 이하는 필터링함 
    # data=heat_data_filtered,    # 데이터 12월1일~21일, 주문수 3건 이하는 필터링함 
    radius=7,        # 점의 확산 반경 (값 클수록 부드럽게)
    blur=7,          # 블러 정도
    min_opacity=0.5,  # 최소 투명도
    max_zoom=13,       # 확대에 따른 반응 정도
    gradient={
        0.2: 'blue',
        0.4: 'cyan',
        0.6: 'lime',
        0.8: 'yellow',
        1.0: 'red'
    }
    ).add_to(m)

radii = [1000, 2000, 3000, 4000, 5000, 6000]
for r in radii:
    folium.Circle(
        location=[37.82151762350629, 127.09344094168883],
        radius=r,
        color="red",
        weight=2,                 # 선 두께
        fill=False,               # 내부 채우기 없음
        dash_array="3, 3",        # 점선 (길이, 간격)
        tooltip=f"{r//1000} km"
    ).add_to(m)

# m
m.save(f"경기도.html")